In [1]:
%load_ext dotenv
%dotenv

# MINE at Retrieval! 
MINE is a tool to evaluate the quality of an LLM for knowledge graph creation. 
It generates a knowledge graph and then tries to retrieve data from it. 

MINER, MINE at Retrieval is like MINE but operating purely at the retrieval stage. 
Knowledge ingestion and retrieval are performed separately to the tool, allowing it to evaluate multiple knowledge storage techniques. 
Put simply, it's one level up in abstraction. 

This can be run for anything that takes some text as an input query and outputs some text containing information related to it. 
We may want to change the dataset a bit, as the queries are not in the form of questions. 


In [2]:
# Implement this for a system
class MINER(object):
	async def ingest(self, text: str):
		""" Ingest knowledge from some text. """
		pass
	async def pre_retrieve(self):
		pass
	async def retrieve(self, text: str) -> str:
		""" Find information relevant to a text. """
		pass
	async def reset(self):
		""" Forget ingested knowledge. """
		pass


In [ ]:
from pathlib import Path
import json
import time
import dspy

MINE_DIRECTORY = Path("../datasets/MINE")


class EvalSignature(dspy.Signature):
	""" 
	ROLE: You are an evaluator that checks if the statement can be deduced from the information in the context.
	TASK: Determine whether the context contains the information stated in the statement.
	"""
	context: str = dspy.InputField()
	statement: str = dspy.InputField()
	context_contains_statement: bool = dspy.OutputField()
eval = dspy.Predict(EvalSignature)

 
def score_count(result):
	""" Computes total score and count. """
	score = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			count += 1
			score += int(query["contained"])
	return score, count


def conciseness(result):
	""" 
	For each "correct" response, how long is it? 
	
	Note: Having only one correct response that is concise will give a "good" score for this. 
	"""
	length = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			if query["contained"]:
				length += len(query["context"])
				count += 1
	return length / count


def mean_median_query_time(result):
	times = []
	for part in result:
		for query in part["queries"]:
			times.append(query["duration"])
	mean = sum(times) / len(times)
	times.sort()
	median = times[len(times)//2]
	return mean, median


# Concurrency is not used here for now
# Maaaybe later 
# If I am annoyed enough! 
# TODO: timing is influenced by caching and I'm not sure what to do about it
async def miner_evaluate_individual(miner: MINER, judge_model: str):
	result = []
	paths = list(MINE_DIRECTORY.iterdir())[:8]
	for i, p in enumerate(paths):
		print(f"Evaluate {p.name} ({i+1}/{len(paths)})")
		# Load data 
		with open(p, "r") as fp:
			mine_data = json.load(fp)
		
		# Ingest text 
		print("Ingest...")
		ingest_st = time.time()
		await miner.ingest(mine_data["essay"])
		await miner.pre_retrieve()
		ingest_en = time.time()

		# Query and evaluate 
		queries = []
		with dspy.context(lm=dspy.LM(judge_model)):
			for i, a in enumerate(mine_data["answers"]):
				print(f"\rQuery {i+1}/{len(mine_data["answers"])}", end="")
				q_st = time.time()
				info = await miner.retrieve(a)
				q_en = time.time()
				contained = (await eval.acall(context=info, statement=a)).context_contains_statement
				queries.append({
					"query": a,
					"context": info,
					"contained": contained,
					"duration": q_en - q_st,
				})
			print()
		result.append({
			"filename": p.name,
			"ingest_duration": ingest_en - ingest_st,
			"queries": queries,
		})
		await miner.reset()
	return result
		

In [4]:
import asyncio
from typing import Any
from aiolimiter import AsyncLimiter
from prettytable import PrettyTable

results_dir = Path("/tmp/miner")


async def evaluate(
	items: list[tuple[str, Any]],
	concurrency: int = 3,
):
	limiter = AsyncLimiter(concurrency)
	async def limited(f):
		async with limiter:
			return await f
	name_to_path = lambda n: results_dir / f"{n}.json"

	names, tasks = zip(*items)
	paths = [name_to_path(n) for n in names]

	print(f"Running {len(names)} evaluations with concurrency {concurrency}")
	if concurrency > 1:
		tasks = [limited(f) for f in tasks]
		results = await asyncio.gather(*tasks)
	else: 
		print("haha that's serial")
		results = [await f for f in tasks]
	print("Done!")

	results_dir.mkdir(exist_ok=True)
	for name, path, result in zip(names, paths, results):
		print(f"Saving '{path}'")
		with open(path, "w") as fp:
			json.dump({
				"name": name,
				"result": result,
			}, fp, indent=2)
	return paths


def show_results():
	table = PrettyTable()
	table.field_names = [
		"Name", 
		"Score", 
		"Context Length", 
		"Conciseness",
		"Query Duration (mean)", 
		"Query Duration (median)",
	]
	results_dir.mkdir(exist_ok=True)
	for f in results_dir.iterdir():
		with open(f, "r") as fp:
			data = json.load(fp)
		score, count = score_count(data["result"])
		r_conciseness = conciseness(data["result"])
		mean, median = mean_median_query_time(data["result"])
		table.add_row([
			data["name"], 
			f"{score/count*100:.2f}% ({score}/{count})", 
			f"{r_conciseness:.2f}",
			f"{score/count*100/r_conciseness:.2f}",
			f"{mean:.2f}s",
			f"{median:.2f}s",
		])
	print(table)
		

# Example Systems
Something to test with, reference implementations. 

In [ ]:
import litellm
import numpy as np


class BasicVectorMINER(MINER):
	""" A MINER implementation for a very simple vector RAG system. """

	chunks: list[str] = []
	embeddings: list = []

	def __init__(
		self,
		chunk_size: int = 200,
		overlap: int = 20,
		quantile: float = 0.95,
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.chunk_size = chunk_size
		self.overlap = overlap
		self.quantile = quantile 
		self.embedding_model = embedding_model

	async def ingest(self, text: str):
		new_chunks = [text[i*self.chunk_size:(i+1)*self.chunk_size+self.overlap] for i in range(0, len(text)//self.chunk_size)]
		new_embeddings = await litellm.aembedding(input=new_chunks, model=self.embedding_model)
		new_embeddings = [np.array(e.embedding) for e in new_embeddings.data]

		self.chunks += new_chunks
		self.embeddings += new_embeddings
	
	async def pre_retrieve(self):
		pass

	async def retrieve(self, text: str) -> str:
		# Make embedding 
		text_embedding = (await litellm.aembedding(self.embedding_model, input=[text]))
		text_embedding = np.array(text_embedding.data[0].embedding)

		# Find similarities 
		cosine = np.abs(np.dot(self.embeddings, text_embedding) / (np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(text_embedding)))
		sims = np.argsort(cosine)[::-1]

		# Find relevant 
		cutoff = np.quantile(sims, self.quantile)
		where = sims >= cutoff
		indices = np.nonzero(where)[0]

		# No need for ordering 
		return "\n".join([self.chunks[i] for i in indices])

	async def forget(self):
		self.chunks = []
		self.embeddings = []

miner = BasicVectorMINER()
result = await miner_evaluate_individual(miner, "bedrock/us.amazon.nova-pro-v1:0")
score_count(result)

judge_model = "bedrock/us.amazon.nova-pro-v1:0"
await evaluate(
	[
		("vector_200_20_95", miner_evaluate_individual(BasicVectorMINER(), judge_model)),
		("vector_200_20_85", miner_evaluate_individual(BasicVectorMINER(quantile=0.85), judge_model)),
		("vector_100_20_95", miner_evaluate_individual(BasicVectorMINER(chunk_size=100), judge_model)),
		("vector_100_20_85", miner_evaluate_individual(BasicVectorMINER(chunk_size=100, quantile=0.85), judge_model)),
	],
	3,
)
show_results()

+------------------+-----------------+----------------+-------------+-----------------------+-------------------------+
|       Name       |      Score      | Context Length | Conciseness | Query Duration (mean) | Query Duration (median) |
+------------------+-----------------+----------------+-------------+-----------------------+-------------------------+
|  hgr_40_50_40_5  | 56.67% (68/120) |    1040.69     |     0.05    |         0.09s         |          0.09s          |
| vector_100_20_85 | 79.17% (95/120) |    15063.97    |     0.01    |         0.15s         |          0.12s          |
| vector_100_20_95 | 23.33% (28/120) |    3009.89     |     0.01    |         0.13s         |          0.11s          |
| vector_200_20_85 | 32.50% (39/120) |    9092.33     |     0.00    |         0.14s         |          0.11s          |
| vector_200_20_95 | 15.83% (19/120) |    3748.84     |     0.00    |         0.14s         |          0.11s          |
|  hgr_60_50_60_5  | 55.83% (67/120) |  

In [8]:

# Implement this for a system
from hypergraph import aggregate, extract_edges_entities, query


class HypergraphMINER(MINER):
	kb = []
	eb = []
	hg = None

	def __init__(
		self,
		# Settings from paper
		kv: int = 60,
		tv: int = 50,
		kh: int = 60,
		th: int = 5,
		# This model doesn't make an error! Many do... 
		model: str = "bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0",
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.kv = kv
		self.tv = tv
		self.kh = kh
		self.th = th
		self.model = model
		self.embedding_model = embedding_model
	
	async def ingest(self, text: str):
		with dspy.context(lm=dspy.LM(self.model)):
			k, e = await extract_edges_entities(text)	
		self.kb += k
		self.eb += e
	
	async def pre_retrieve(self):
		with dspy.context(lm=dspy.LM(self.model)):
			self.hg = await aggregate(self.kb, self.eb, self.embedding_model)

	async def retrieve(self, text: str) -> str:
		assert not (self.hg is None)
		with dspy.context(lm=dspy.LM(self.model)):
			k = await query(text, self.hg, self.embedding_model, kv=self.kv, tv=self.tv, kh=self.kh, th=self.th)
		return k

	async def reset(self):
		self.kb = []
		self.eb = []
		self.hg = None

await evaluate(
	[
		# ("hgr_60_50_60_5", miner_evaluate_individual(HypergraphMINER(60, 50, 60, 5), judge_model)),
		("hgr_60_30_60_3", miner_evaluate_individual(HypergraphMINER(60, 30, 60, 3), judge_model)),
	],
	1,
)
show_results()

Running 1 evaluations with concurrency 1
haha that's serial
Evaluate A Brief History of Time Zones.json (1/8)
Ingest...
Aggregate into graph
Generate embeddings
Add entities
Add edges
Query 1/15Found 10 entities
Found 10 hyperedges
Selected 4 entities
Selected 6 hyperedges
Have 10 information pieces
Fusion expand to 11 information pieces
Query 2/15Found 10 entities
Found 10 hyperedges
Selected 6 entities
Selected 9 hyperedges
Have 15 information pieces
Fusion expand to 20 information pieces
Query 3/15Found 10 entities
Found 10 hyperedges
Selected 5 entities
Selected 5 hyperedges
Have 10 information pieces
Fusion expand to 12 information pieces
Query 4/15Found 10 entities
Found 10 hyperedges
Selected 3 entities
Selected 7 hyperedges
Have 10 information pieces
Fusion expand to 14 information pieces
Query 5/15Found 10 entities
Found 10 hyperedges
Selected 1 entities
Selected 4 hyperedges
Have 5 information pieces
Fusion expand to 13 information pieces
Query 6/15Found 10 entities
Found 10 

In [10]:
show_results()


+------------------+------------------+----------------+-------------+-----------------------+-------------------------+
|       Name       |      Score       | Context Length | Conciseness | Query Duration (mean) | Query Duration (median) |
+------------------+------------------+----------------+-------------+-----------------------+-------------------------+
|  hgr_60_30_60_3  | 95.83% (115/120) |    1780.73     |     0.05    |         0.10s         |          0.10s          |
| vector_100_20_85 | 79.17% (95/120)  |    15063.97    |     0.01    |         0.15s         |          0.12s          |
| vector_100_20_95 | 23.33% (28/120)  |    3009.89     |     0.01    |         0.13s         |          0.11s          |
| vector_200_20_85 | 32.50% (39/120)  |    9092.33     |     0.00    |         0.14s         |          0.11s          |
| vector_200_20_95 | 15.83% (19/120)  |    3748.84     |     0.00    |         0.14s         |          0.11s          |
|  hgr_60_50_60_5  | 80.83% (97/

In [ ]:



class KGv2MINER(MINER):
	kb = []
	eb = []
	hg = None

	def __init__(
		self,
		# Settings from paper
		kv: int = 60,
		tv: int = 50,
		kh: int = 60,
		th: int = 5,
		# This model doesn't make an error! Many do... 
		model: str = "bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0",
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.kv = kv
		self.tv = tv
		self.kh = kh
		self.th = th
		self.model = model
		self.embedding_model = embedding_model
	
	async def ingest(self, text: str):
		with dspy.context(lm=dspy.LM(self.model)):
			k, e = await extract_edges_entities(text)	
		self.kb += k
		self.eb += e
	
	async def pre_retrieve(self):
		with dspy.context(lm=dspy.LM(self.model)):
			self.hg = await aggregate(self.kb, self.eb, self.embedding_model)

	async def retrieve(self, text: str) -> str:
		assert not (self.hg is None)
		with dspy.context(lm=dspy.LM(self.model)):
			k = await query(text, self.hg, self.embedding_model, kv=self.kv, tv=self.tv, kh=self.kh, th=self.th)
		return k

	async def reset(self):
		self.kb = []
		self.eb = []
		self.hg = None

await evaluate(
	[
		# ("hgr_60_50_60_5", miner_evaluate_individual(HypergraphMINER(60, 50, 60, 5), judge_model)),
		("hgr_60_30_60_3", miner_evaluate_individual(HypergraphMINER(60, 30, 60, 3), judge_model)),
	],
	1,
)
show_results()